# African Gig-Economy & Digital Wallet Risk — EDA & Statistical Verification

This notebook reproduces the full analysis behind `Analytical_Report.docx`: structural EDA, chart generation, and the statistical claim-verification battery that tests the original brief's six hypothesized risk patterns against the actual data.

**Headline finding:** zero of six claims held up (all p > 0.05). See Section 4 below for the full test-by-test breakdown.

**How to use this notebook:**
1. Run the Setup cell to install/import dependencies.
2. Run the Data loading cell — it will prompt you to upload the five source CSVs if they aren't already present in `/content/data/`.
3. Run the remaining cells top to bottom.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
    'figure.facecolor': 'white', 'axes.facecolor': 'white'
})
NAVY, TEAL, CORAL, GOLD, GREY, BLUE = '#1B2A4A', '#0E7C7B', '#E8604C', '#E3B23C', '#8A94A6', '#3E6FE0'


## 2. Data loading

This notebook expects five CSVs: `dim_channel.csv`, `dim_date_updated.csv`, `dim_market.csv`, `dim_worker.csv`, `fact_transactions_Updated_.csv`.

- **Running in Colab with the repo cloned or Drive-mounted:** set `DATA_DIR` to wherever `data/` ended up (e.g. `/content/gig_wallet_risk_project/data`).
- **Running standalone in Colab:** leave `DATA_DIR` as `/content/data`, run the cell, and use the upload widget that appears if the files aren't found.

In [ ]:
DATA_DIR = '/content/data'  # change this if your data folder is elsewhere

os.makedirs(DATA_DIR, exist_ok=True)
required_files = [
    'dim_channel.csv', 'dim_date_updated.csv', 'dim_market.csv',
    'dim_worker.csv', 'fact_transactions_Updated_.csv'
]
missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    try:
        from google.colab import files
        print(f"Missing {len(missing)} file(s) in {DATA_DIR}: {missing}")
        print("Upload them now (multi-select all five CSVs in the file picker):")
        uploaded = files.upload()
        for fname, content in uploaded.items():
            with open(os.path.join(DATA_DIR, fname), 'wb') as f:
                f.write(content)
        print("Upload complete.")
    except ImportError:
        raise FileNotFoundError(
            f"Missing files {missing} in {DATA_DIR} and google.colab isn't available "
            "to prompt an upload. Place the CSVs in DATA_DIR manually and re-run this cell."
        )
else:
    print("All 5 source files found in", DATA_DIR)


In [ ]:
fact = pd.read_csv(f'{DATA_DIR}/fact_transactions_Updated_.csv')
worker = pd.read_csv(f'{DATA_DIR}/dim_worker.csv')
channel = pd.read_csv(f'{DATA_DIR}/dim_channel.csv')
market = pd.read_csv(f'{DATA_DIR}/dim_market.csv')
date = pd.read_csv(f'{DATA_DIR}/dim_date_updated.csv')

df = (fact.merge(worker, on='worker_id', how='left')
          .merge(channel, on='channel_id', how='left')
          .merge(market, on='market_id', how='left')
          .merge(date, on='date_id', how='left'))

df['full_date'] = pd.to_datetime(df['full_date'], format='%m/%d/%Y')
df['tenure_bucket'] = pd.cut(
    df.account_tenure_days, [-1, 90, 365, 730, 1095, 1460, 1825],
    labels=['0-90d', '91-365d', '1-2yr', '2-3yr', '3-4yr', '4-5yr']
)
df['ym'] = df['full_date'].dt.to_period('M').astype(str)

print("Merged shape:", df.shape)
df.head()


## 3. Structural EDA — schema, nulls, uniqueness, referential integrity

In [ ]:
for name, d in [('fact', fact), ('worker', worker), ('channel', channel), ('market', market), ('date', date)]:
    print(f"=== {name}  shape={d.shape} ===")
    nulls = d.isna().sum()
    print("nulls:", dict(nulls[nulls > 0]) if nulls.sum() else "none")
    print()

print("Duplicate transaction_id:", fact['transaction_id'].duplicated().sum())
print("worker_id not in dim_worker:", (~fact.worker_id.isin(worker.worker_id)).sum())
print("channel_id not in dim_channel:", (~fact.channel_id.isin(channel.channel_id)).sum())
print("market_id not in dim_market:", (~fact.market_id.isin(market.market_id)).sum())
print("date_id not in dim_date:", (~fact.date_id.isin(date.date_id)).sum())


In [ ]:
print("Total transactions:", len(df))
print("Total value USD:", df.amount_usd.sum())
print("Total fraud loss USD:", df.fraud_loss_usd.sum())
print("Fraud-flagged rate:", df.is_fraud_flagged.mean())
print("Disputed rate:", df.is_disputed.mean())
print("Reversed rate:", df.is_reversed.mean())

# fraud_loss_usd is mechanically derived from is_fraud_flagged, not an independent signal
print("\nfraud_loss_usd == 0 when flagged:", ((df.is_fraud_flagged) & (df.fraud_loss_usd == 0)).sum(), "/", df.is_fraud_flagged.sum())
print("fraud_loss_usd > 0 when NOT flagged:", ((~df.is_fraud_flagged) & (df.fraud_loss_usd > 0)).sum(), "/", (~df.is_fraud_flagged).sum())


## 4. Statistical claim verification

Each of the six claims from the original brief, tested directly: chi-square test of independence for categorical splits, Pearson correlation for the velocity claim. A result is only called CONFIRMED where p < 0.05 **and** the effect direction/size matches the claim.

In [ ]:
def chi2_test(data, col, target='is_fraud_flagged'):
    ct = pd.crosstab(data[col], data[target])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    return chi2, p

results = []

chi2, p = chi2_test(df, 'channel_type', 'is_fraud_flagged')
results.append(("USSD shows 2.3x higher fraud rate than app channel", chi2, p))

chi2, p = chi2_test(df, 'country', 'is_fraud_flagged')
results.append(("Nigeria + Kenya drive 70% of fraud volume", chi2, p))

chi2, p = chi2_test(df, 'tenure_bucket', 'is_fraud_flagged')
results.append(("New accounts (<90d) show 3x higher fraud rate", chi2, p))

chi2, p = chi2_test(df, 'gig_segment', 'is_disputed')
results.append(("Market traders have highest dispute rate among gig segments", chi2, p))

chi2, p = chi2_test(df, 'is_month_end', 'is_reversed')
results.append(("Month-end spikes in cash-out and reversal rates", chi2, p))

print(f"{'Claim':<55} {'chi2':>8} {'p-value':>10}  Verdict")
print("-" * 90)
for label, chi2, p in results:
    verdict = "CONFIRMED" if p < 0.05 else "NOT CONFIRMED"
    print(f"{label:<55} {chi2:>8.2f} {p:>10.4f}  {verdict}")

r_fraud = df['velocity_score'].corr(df['is_fraud_flagged'].astype(int))
r_rev = df['velocity_score'].corr(df['is_reversed'].astype(int))
verdict = "CONFIRMED" if (abs(r_fraud) > 0.1 or abs(r_rev) > 0.1) else "NOT CONFIRMED"
print(f"\nHigh velocity score correlates with fraud/reversal: r(fraud)={r_fraud:.4f}  r(reversal)={r_rev:.4f}  {verdict}")


In [ ]:
# Full numeric correlation matrix
numcols = ['amount_usd', 'velocity_score', 'fraud_loss_usd', 'processing_time_ms', 'account_tenure_days', 'risk_score']
tmp = df[numcols + ['is_fraud_flagged', 'is_disputed', 'is_reversed']].copy()
for c in ['is_fraud_flagged', 'is_disputed', 'is_reversed']:
    tmp[c] = tmp[c].astype(int)

corr = tmp.corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-0.3, vmax=0.3, ax=ax, cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation matrix — near-zero except the mechanical loss/flag pair', fontsize=11.5, fontweight='bold', loc='left')
plt.tight_layout()
plt.show()


## 5. Charts

The same  charts used in the analytical report and Power BI dashboard, generated inline here so you can inspect or modify them directly in Colab.

In [ ]:
# Monthly fraud rate trend
m = df.groupby('ym')['is_fraud_flagged'].agg(['mean', 'count'])
m['se'] = np.sqrt(m['mean'] * (1 - m['mean']) / m['count'])
overall = df['is_fraud_flagged'].mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
x = range(len(m))
ax.plot(x, m['mean'] * 100, color=NAVY, lw=2, marker='o', ms=4, label='Monthly fraud rate')
ax.fill_between(x, (m['mean'] - 1.96 * m['se']) * 100, (m['mean'] + 1.96 * m['se']) * 100, color=NAVY, alpha=0.15, label='95% CI')
ax.axhline(overall * 100, color=CORAL, ls='--', lw=1.5, label=f'Overall mean ({overall*100:.1f}%)')
ax.set_xticks(x); ax.set_xticklabels(m.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Fraud-flagged rate (%)')
ax.set_title('Monthly fraud rate — no trend or seasonality', fontsize=12, fontweight='bold', loc='left')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Fraud rate by channel type, with 95% CI
g = df.groupby('channel_type')['is_fraud_flagged'].agg(['mean', 'count']).sort_values('mean')
g['se'] = np.sqrt(g['mean'] * (1 - g['mean']) / g['count'])

fig, ax = plt.subplots(figsize=(8, 4.5))
y = range(len(g))
ax.barh(y, g['mean'] * 100, xerr=g['se'] * 196, color=TEAL, alpha=0.85, capsize=4)
ax.axvline(overall * 100, color=CORAL, ls='--', lw=1.5, label=f'Overall mean ({overall*100:.1f}%)')
ax.set_yticks(y); ax.set_yticklabels(g.index)
ax.set_xlabel('Fraud-flagged rate (%)')
ax.set_title('Fraud rate by channel type — differences fall within noise', fontsize=12, fontweight='bold', loc='left')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Fraud rate by country
g = df.groupby('country')['is_fraud_flagged'].agg(['mean', 'count']).sort_values('mean')
g['se'] = np.sqrt(g['mean'] * (1 - g['mean']) / g['count'])

fig, ax = plt.subplots(figsize=(8, 4))
y = range(len(g))
ax.barh(y, g['mean'] * 100, xerr=g['se'] * 196, color=NAVY, alpha=0.85, capsize=4)
ax.axvline(overall * 100, color=CORAL, ls='--', lw=1.5, label=f'Overall mean ({overall*100:.1f}%)')
ax.set_yticks(y); ax.set_yticklabels(g.index)
ax.set_xlabel('Fraud-flagged rate (%)')
ax.set_title('Fraud rate by country — not statistically distinguishable', fontsize=12, fontweight='bold', loc='left')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Fraud rate by tenure group
g = df.groupby('account_tenure_group', observed=True)['is_fraud_flagged'].agg(['mean', 'count'])
g['se'] = np.sqrt(g['mean'] * (1 - g['mean']) / g['count'])

fig, ax = plt.subplots(figsize=(8, 4.5))
x = range(len(g))
ax.bar(x, g['mean'] * 100, yerr=g['se'] * 196, color=GOLD, alpha=0.9, capsize=4, edgecolor=NAVY)
ax.axhline(overall * 100, color=CORAL, ls='--', lw=1.5, label=f'Overall mean ({overall*100:.1f}%)')
ax.set_xticks(x); ax.set_xticklabels(g.index)
ax.set_ylabel('Fraud-flagged rate (%)')
ax.set_title('Fraud rate by account tenure — new accounts are not higher risk here', fontsize=12, fontweight='bold', loc='left')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Transaction value by type
g = df.groupby('transaction_type').agg(count=('transaction_id', 'count'), total_usd=('amount_usd', 'sum')).sort_values('total_usd')

fig, ax = plt.subplots(figsize=(9, 6))
y = range(len(g))
ax.barh(y, g['total_usd'] / 1e6, color=TEAL)
ax.set_yticks(y); ax.set_yticklabels(g.index, fontsize=9)
ax.set_xlabel('Total value (USD millions)')
ax.set_title('Transaction value by type — evenly distributed across 13 types', fontsize=12, fontweight='bold', loc='left')
plt.tight_layout()
plt.show()


In [ ]:
# Fraud loss exposure by country
g = df.groupby('country').agg(fraud_loss=('fraud_loss_usd', 'sum'), total_value=('amount_usd', 'sum')).sort_values('fraud_loss')

fig, ax = plt.subplots(figsize=(8, 4))
y = range(len(g))
ax.barh(y, g['fraud_loss'] / 1e6, color=CORAL, alpha=0.85)
ax.set_yticks(y); ax.set_yticklabels(g.index)
ax.set_xlabel('Total fraud loss exposure (USD millions)')
ax.set_title('Fraud loss exposure by country — proportional to volume', fontsize=11.5, fontweight='bold', loc='left')
plt.tight_layout()
plt.show()


## 6. Summary

Zero of six brief claims were statistically confirmed (all p > 0.05). See `Analytical_Report.docx` Section 3 for the full write-up, `powerbi_model/` for the Power BI dashboard build, and `powerbi_model/Claim_Verification.csv` for the exact table that feeds the dashboard's verdict table.

If you're running this against a different or updated data extract, re-run Section 4 above unchanged — it's designed as a repeatable validation harness, not a one-off critique of this specific file.